# C17a — Évaluation de la transcription (EF-10, E1, P10, P12)

Deux modèles faster-whisper (base et small, révisions figées dans `configs/transcription.yaml`),
en français, sur deux corpus d'OpenSLR 57 lus depuis le manifeste (P10) :

- le **test officiel** (515 phrases, 12 locuteurs, 0,31 h) ;
- la partie lue du sous-corpus de **Yaoundé**, `yaounde/read` (6 299 phrases, 84 locuteurs,
  8,53 h), rangée dans `train` mais jamais utilisée pour entraîner. La partie `yaounde/answers`
  est exclue : sa transcription dans le manifeste est la question posée, pas la réponse
  enregistrée.

Référence et transcription sont normalisées de la même façon (P12) avant le WER et le CER.
Intervalles à 95 % par bootstrap **par locuteur**. Puis une vérification sur CPU en int8 (100
phrases), et un audio de 30 s pour mesurer la latence sur la machine de référence.

Évaluation seulement : rien n'est branché dans l'application. Internet activé.

**Deux passages.**
1. Premier passage (GPU T4) : transcription float16 de 8 912 phrases (test et tout Yaoundé),
   enregistrée dans `c17_hypotheses_float16.json`.
2. Second passage (CPU seul) : la sortie du premier passage est ajoutée en entrée du notebook.
   Les transcriptions déjà là ne sont pas recalculées : aucun calcul sur GPU. Le WER est
   recalculé sur test et `yaounde/read`, et la vérification int8 est refaite.

## a. Démarrage

On clone le dépôt (branche `BRANCH`), on l'installe en mode éditable (I9), puis faster-whisper,
CTranslate2 et jiwer aux versions de `uv.lock`, et cuBLAS et cuDNN pour CUDA 12 (voir e).

In [ ]:
import sys

BRANCH = "exp/c17-transcription"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}
%pip install -q "faster-whisper==1.2.1" "ctranslate2==4.8.5" "jiwer==4.0.0"
# CTranslate2 4.8.5 est compilé pour CUDA 12 : premier essai, « libcublas.so.12 is not found ».
%pip install -q "nvidia-cublas-cu12==12.9.3.13" "nvidia-cudnn-cu12==9.28.1.4"

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. ffmpeg et GPU

`load_audio` décode avec ffmpeg (D1). Le GPU n'est exigé que s'il reste des phrases à transcrire
en float16 (étape f).

In [ ]:
import shutil
import time

import ctranslate2
import faster_whisper

if shutil.which("ffmpeg") is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg
gpu_count = ctranslate2.get_cuda_device_count()
FASTER_WHISPER_VERSION = faster_whisper.__version__
CTRANSLATE2_VERSION = ctranslate2.__version__
print("GPU vus par CTranslate2 :", gpu_count)
print("faster-whisper", FASTER_WHISPER_VERSION, "· CTranslate2", CTRANSLATE2_VERSION)
NOTEBOOK_START = time.time()

## c. OpenSLR 57

Archive officielle (environ 1,8 Go), décompressée dans `/tmp` pour que l'audio ne fasse pas
partie des sorties du notebook. Lien `/kaggle/working/datasets/openslr57` (contrat des
manifestes, docs/datasets.md).

In [ ]:
import tarfile
import urllib.request
from pathlib import Path

from echolens.core.data.manifest import dataset_root

OPENSLR57_TAR_URL = "https://openslr.elda.org/resources/57/African_Accented_French.tar.gz"
DOWNLOAD_DIR = Path("/tmp/openslr57")
DATASETS_DIR = Path("/kaggle/working/datasets")


def download_openslr57() -> Path:
    # La racine du jeu est le dossier African_Accented_French (speech/ et transcripts/).
    archive_path = DOWNLOAD_DIR / "openslr57.tar.gz"
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(OPENSLR57_TAR_URL, archive_path)
    with tarfile.open(archive_path) as archive:
        archive.extractall(DOWNLOAD_DIR, filter="data")
    archive_path.unlink()
    candidates = sorted(DOWNLOAD_DIR.glob("**/African_Accented_French"))
    return candidates[0]


openslr57_root = download_openslr57()
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
link = dataset_root("openslr57", DATASETS_DIR)
if link.is_symlink():
    link.unlink()
link.symlink_to(openslr57_root)
print(link, "→", openslr57_root)

## d. Corpus lus depuis le manifeste (P10)

Transcriptions et locuteurs viennent du manifeste versionné. On vérifie les effectifs attendus
et la présence de chaque fichier audio.

In [ ]:
from echolens.config import REPO_ROOT
from echolens.core.data.manifest import audio_path, load_manifest
from echolens.training.transcription_eval import corpus_rows, load_transcription_eval_config

EXPECTED_SENTENCES = {"test": 515, "yaounde_read": 6299}

config = load_transcription_eval_config()
manifest = load_manifest(REPO_ROOT / "data" / "manifests" / "openslr57.csv")
rows_by_corpus = {}
for selection in config.corpora:
    rows = corpus_rows(manifest, selection)
    if len(rows) != EXPECTED_SENTENCES[selection.id]:
        raise RuntimeError(
            f"{selection.id} : {len(rows)} phrases au lieu de {EXPECTED_SENTENCES[selection.id]}"
        )
    missing = [row.path for row in rows if not audio_path(row, DATASETS_DIR).exists()]
    if missing:
        raise RuntimeError(
            f"{selection.id} : {len(missing)} fichiers absents, par exemple {missing[0]}"
        )
    rows_by_corpus[selection.id] = rows
    speakers = {row.speaker for row in rows}
    hours = sum(row.duration_s for row in rows) / 3600
    print(f"{selection.id} : {len(rows)} phrases, {len(speakers)} locuteurs, {hours:.2f} h")

## e. Transcriptions float16 déjà enregistrées

Si la sortie du premier passage est en entrée du notebook, on reprend son
`c17_hypotheses_float16.json` (transcriptions brutes de chaque phrase par chaque modèle) et
le temps GPU qu'il avait mesuré. On compte les phrases de test et de `yaounde/read` qui n'ont
pas encore de transcription.

In [ ]:
import json

OUTPUT_DIR = Path("/kaggle/working")
HYPOTHESES_PATH = OUTPUT_DIR / "c17_hypotheses_float16.json"
INPUT_DIR = Path("/kaggle/input")

# Étape 1 : copie des transcriptions du premier passage, si elles sont en entrée.
previous_hypotheses = sorted(INPUT_DIR.glob("**/c17_hypotheses_float16.json"))
if previous_hypotheses and not HYPOTHESES_PATH.exists():
    shutil.copy(previous_hypotheses[0], HYPOTHESES_PATH)
    print("Transcriptions reprises de", previous_hypotheses[0])
hypotheses = {model.id: {} for model in config.models}
if HYPOTHESES_PATH.exists():
    hypotheses = json.loads(HYPOTHESES_PATH.read_text(encoding="utf-8"))

# Étape 2 : temps GPU du premier passage (8 912 phrases, answers comprises).
gpu_seconds = {model.id: 0.0 for model in config.models}
previous_results = sorted(INPUT_DIR.glob("**/c17_transcription.json"))
if previous_results:
    previous = json.loads(previous_results[0].read_text(encoding="utf-8"))
    gpu_seconds = previous["gpu_seconds"]

# Étape 3 : phrases sans transcription pour au moins un modèle.
all_rows = rows_by_corpus["test"] + rows_by_corpus["yaounde_read"]
missing_rows = []
for row in all_rows:
    for model in config.models:
        if row.path not in hypotheses[model.id]:
            missing_rows.append(row)
            break
print(
    len(all_rows) - len(missing_rows),
    "phrases déjà transcrites ;",
    len(missing_rows),
    "à transcrire",
)

## f. Transcription sur GPU (float16), seulement pour les phrases manquantes

**Bibliothèques CUDA 12.** Au premier essai, l'image Kaggle ne fournissait pas
`libcublas.so.12`, que demande CTranslate2 4.8.5 (compilé pour CUDA 12). On garde le même
moteur : cuBLAS et cuDNN pour CUDA 12 viennent des paquets pip `nvidia-cublas-cu12` et
`nvidia-cudnn-cu12` (méthode indiquée par faster-whisper), chargés par leur chemin avant le
premier calcul. Puis on charge base sur GPU en float16 et on transcrit 1 s de silence. Si
CTranslate2 ne trouve toujours pas cuBLAS ou cuDNN, **on s'arrête ici** : pas de changement
de moteur sans décision de l'équipe.

Chaque phrase est décodée une fois (`load_audio`, mono 16 kHz) puis transcrite par les deux
modèles, avec un enregistrement tous les 500 fichiers : une relance reprend où le calcul
s'était arrêté. **S'il ne manque aucune phrase, rien ne tourne sur GPU.**

In [ ]:
import ctypes
import importlib.util
from importlib.metadata import version

import numpy as np

from echolens.core.audio import load_audio
from echolens.training.transcription_models import load_whisper, transcribe

SAVE_EVERY = 500
CUDA_LIBRARIES = {
    "nvidia-cublas-cu12": version("nvidia-cublas-cu12"),
    "nvidia-cudnn-cu12": version("nvidia-cudnn-cu12"),
}


def nvidia_library_dir(package: str) -> Path:
    # Les paquets nvidia-* installent leurs .so dans site-packages/nvidia/<nom>/lib.
    spec = importlib.util.find_spec(package)
    return Path(spec.submodule_search_locations[0]) / "lib"


def preload_libraries(directory: Path) -> list[str]:
    # Chargement global (RTLD_GLOBAL), en plusieurs passes : une bibliothèque du dossier peut
    # dépendre d'une autre qui n'est pas encore chargée. Renvoie celles qui ont échoué.
    pending = sorted(directory.glob("lib*.so*"))
    for _ in range(len(pending)):
        failed = []
        for path in pending:
            try:
                ctypes.CDLL(str(path), mode=ctypes.RTLD_GLOBAL)
            except OSError:
                failed.append(path)
        if len(failed) == len(pending):
            break
        pending = failed
    return [path.name for path in pending]


def prepare_gpu():
    # GPU visible, bibliothèques CUDA 12 chargées, puis essai sur 1 s de silence.
    if gpu_count == 0:
        raise RuntimeError("CTranslate2 ne voit aucun GPU : activer l'accélérateur GPU T4.")
    for package in ["nvidia.cublas", "nvidia.cudnn"]:
        not_loaded = preload_libraries(nvidia_library_dir(package))
        print(package, "non chargées :", not_loaded)
    models_by_id = {model.id: model for model in config.models}
    try:
        probe = load_whisper(models_by_id["base"], "cuda", "float16", config)
        transcribe(probe, np.zeros(16_000, dtype=np.float32), config)
    except Exception as error:
        raise RuntimeError(
            "CTranslate2 ne trouve pas les bibliothèques CUDA de Kaggle "
            f"({type(error).__name__} : {error}). Arrêt, sans changer de moteur."
        ) from error
    print("CTranslate2 fonctionne sur le GPU en float16.")


def transcribe_on_gpu(rows):
    # Les deux modèles transcrivent chaque phrase ; enregistrement régulier pour la reprise.
    gpu_models = {}
    for model in config.models:
        gpu_models[model.id] = load_whisper(model, "cuda", "float16", config)
    for index, row in enumerate(rows):
        signal = load_audio(audio_path(row, DATASETS_DIR))
        for model_id, whisper in gpu_models.items():
            start = time.perf_counter()
            hypotheses[model_id][row.path] = transcribe(whisper, signal, config)
            gpu_seconds[model_id] += time.perf_counter() - start
        if (index + 1) % SAVE_EVERY == 0:
            HYPOTHESES_PATH.write_text(json.dumps(hypotheses, ensure_ascii=False), encoding="utf-8")
            print(
                f"{index + 1} / {len(rows)} phrases", {k: round(v) for k, v in gpu_seconds.items()}
            )
    HYPOTHESES_PATH.write_text(json.dumps(hypotheses, ensure_ascii=False), encoding="utf-8")


if missing_rows:
    prepare_gpu()
    transcribe_on_gpu(missing_rows)
else:
    print("Toutes les transcriptions float16 sont déjà là : aucun calcul sur GPU.")
print("Temps GPU par modèle (s) :", {k: round(v) for k, v in gpu_seconds.items()})

## g. WER et CER, intervalles par locuteur, écart small − base

Normalisation P12 des deux côtés, puis erreurs comptées par jiwer. Intervalles à 95 % par
bootstrap par locuteur (1 000 tirages, graine 42). L'écart small − base est apparié : les mêmes
locuteurs sont tirés pour les deux modèles ; s'il contient 0, il n'est pas significatif.

In [ ]:
from echolens.training.transcription_eval import (
    bootstrap_intervals,
    interval_contains_zero,
    paired_wer_difference,
    score_sentence,
)


def score_rows(rows, model_hypotheses):
    scores = []
    for row in rows:
        hypothesis = model_hypotheses[row.path]
        scores.append(score_sentence(row.path, row.speaker, row.transcript, hypothesis))
    return scores


def interval_record(interval):
    return {
        "value": round(interval.value, 4),
        "low": round(interval.low, 4),
        "high": round(interval.high, 4),
    }


scores = {}
results_by_model = {}
for model in config.models:
    scores[model.id] = {}
    results_by_model[model.id] = {}
    for corpus_id, rows in rows_by_corpus.items():
        corpus_scores = score_rows(rows, hypotheses[model.id])
        scores[model.id][corpus_id] = corpus_scores
        wer, cer = bootstrap_intervals(
            corpus_scores, config.bootstrap_draws, config.seed, config.confidence_level
        )
        results_by_model[model.id][corpus_id] = {
            "wer": interval_record(wer),
            "cer": interval_record(cer),
        }
        print(
            f"{model.id:6s} {corpus_id:8s} WER {wer.value:.1%} [{wer.low:.1%} ; {wer.high:.1%}]"
            f"  CER {cer.value:.1%} [{cer.low:.1%} ; {cer.high:.1%}]"
        )

small_minus_base = {}
for corpus_id in rows_by_corpus:
    difference = paired_wer_difference(
        scores["base"][corpus_id],
        scores["small"][corpus_id],
        config.bootstrap_draws,
        config.seed,
        config.confidence_level,
    )
    record = interval_record(difference)
    record["significant"] = not interval_contains_zero(difference)
    small_minus_base[corpus_id] = record
    bounds = f"[{difference.low:+.1%} ; {difference.high:+.1%}]"
    print(f"small − base, {corpus_id} : {difference.value:+.1%}", bounds)

## h. Vérification sur CPU en int8

100 phrases tirées au hasard (graine 42) parmi test et `yaounde/read`, transcrites sur CPU en
int8 par les deux modèles. Écart de WER apparié int8 − float16 sur ces mêmes phrases. Les
transcriptions int8 sont gardées dans `c17_hypotheses_int8.json`.

In [ ]:
import os

from echolens.training.transcription_eval import error_rates, sample_rows

INT8_HYPOTHESES_PATH = OUTPUT_DIR / "c17_hypotheses_int8.json"
check_rows = sample_rows(all_rows, config.cpu_check_sentences, config.seed)
check_paths = [row.path for row in check_rows]
print("CPU :", os.cpu_count(), "fils visibles ;", config.cpu_threads, "utilisés")

int8_minus_float16 = {}
cpu_hypotheses = {}
for model in config.models:
    cpu_model = load_whisper(model, "cpu", "int8", config)
    cpu_hypotheses[model.id] = {}
    start = time.perf_counter()
    for row in check_rows:
        signal = load_audio(audio_path(row, DATASETS_DIR))
        cpu_hypotheses[model.id][row.path] = transcribe(cpu_model, signal, config)
    cpu_seconds = time.perf_counter() - start
    float16_scores = score_rows(check_rows, hypotheses[model.id])
    int8_scores = score_rows(check_rows, cpu_hypotheses[model.id])
    difference = paired_wer_difference(
        float16_scores, int8_scores, config.bootstrap_draws, config.seed, config.confidence_level
    )
    float16_wer, _ = error_rates(float16_scores)
    int8_wer, _ = error_rates(int8_scores)
    record = interval_record(difference)
    record["significant"] = not interval_contains_zero(difference)
    record["float16_wer"] = round(float16_wer, 4)
    record["int8_wer"] = round(int8_wer, 4)
    record["cpu_seconds"] = round(cpu_seconds, 1)
    int8_minus_float16[model.id] = record
    del cpu_model
    print(
        f"{model.id} : float16 {float16_wer:.1%}, int8 {int8_wer:.1%}, "
        f"int8 − float16 {difference.value:+.1%} [{difference.low:+.1%} ; {difference.high:+.1%}], "
        f"{cpu_seconds:.0f} s sur CPU"
    )
INT8_HYPOTHESES_PATH.write_text(json.dumps(cpu_hypotheses, ensure_ascii=False), encoding="utf-8")

## i. Les 20 pires phrases

Par modèle et par corpus, les phrases au plus grand nombre d'erreurs de mots, avec la référence
et la transcription normalisées (P12).

In [ ]:
from echolens.training.transcription_eval import worst_sentences

worst = {}
for model_id, corpus_scores in scores.items():
    worst[model_id] = {}
    for corpus_id, sentence_scores in corpus_scores.items():
        selected = worst_sentences(sentence_scores, config.worst_sentence_count)
        records = []
        for score in selected:
            records.append(
                {
                    "path": score.path,
                    "speaker": score.speaker,
                    "word_errors": score.word_errors,
                    "word_count": score.word_count,
                    "reference": score.reference,
                    "hypothesis": score.hypothesis,
                }
            )
        worst[model_id][corpus_id] = records
first = worst["small"]["test"][0]
print("Pire phrase (small, test) :", first["word_errors"], "erreurs")
print("  référence     :", first["reference"])
print("  transcription :", first["hypothesis"])

## j. Références suspectes du test officiel

Phrases du test où base **et** small ont un WER de phrase ≥ 100 % (`suspect_min_wer`). Deux
modèles qui échouent autant sur la même phrase pointent plutôt vers la référence que vers la
reconnaissance. Elles sont listées avec la référence du manifeste et les transcriptions brutes,
pour être écoutées ; elles **restent dans le calcul**.

In [ ]:
from echolens.training.transcription_eval import suspect_sentences

suspects = suspect_sentences(
    scores["base"]["test"], scores["small"]["test"], config.suspect_min_wer
)
test_rows_by_path = {row.path: row for row in rows_by_corpus["test"]}
suspect_references = []
for base_score, small_score in suspects:
    row = test_rows_by_path[base_score.path]
    suspect_references.append(
        {
            "path": row.path,
            "speaker": row.speaker,
            "reference": row.transcript,
            "base": hypotheses["base"][row.path],
            "small": hypotheses["small"][row.path],
            "word_count": base_score.word_count,
            "base_word_errors": base_score.word_errors,
            "small_word_errors": small_score.word_errors,
        }
    )
    print(row.path)
    print("  référence :", row.transcript)
    print("  base      :", hypotheses["base"][row.path])
    print("  small     :", hypotheses["small"][row.path])
print(len(suspect_references), "phrases suspectes sur", len(rows_by_corpus["test"]))

## k. Audio de 30 s pour la latence sur la machine de référence

Phrases de `yaounde/read` tirées au hasard (graine 42), mises bout à bout jusqu'à 30 s, puis coupées à
30 s exactement. Fichier `c17_latency_30s.wav` en sortie du notebook, jamais commité : le script
`training/scripts/measure_transcription_latency.py` le transcrit sur la machine de référence.

In [ ]:
import soundfile as sf

from echolens.core.audio.io import TARGET_SAMPLE_RATE

LATENCY_WAV_PATH = OUTPUT_DIR / "c17_latency_30s.wav"
target_samples = int(config.latency_audio_s * TARGET_SAMPLE_RATE)
pieces = []
used_paths = []
collected = 0
for row in sample_rows(rows_by_corpus["yaounde_read"], 50, config.seed):
    signal = load_audio(audio_path(row, DATASETS_DIR))
    pieces.append(signal)
    used_paths.append(row.path)
    collected += len(signal)
    if collected >= target_samples:
        break
latency_audio = np.concatenate(pieces)[:target_samples]
sf.write(LATENCY_WAV_PATH, latency_audio, TARGET_SAMPLE_RATE, subtype="PCM_16")
print(
    LATENCY_WAV_PATH,
    f"{len(latency_audio) / TARGET_SAMPLE_RATE:.1f} s,",
    len(used_paths),
    "phrases",
)

## l. Résultats

`c17_transcription.json` (→ `training/results/`) : WER et CER avec intervalles par modèle et
par corpus, écart small − base, écart int8 − float16, 20 pires phrases, références suspectes,
versions, révisions des modèles, temps GPU du premier passage, durée de ce passage. Le champ
`latency_cpu` est ajouté ensuite par le script de latence.

In [ ]:
import platform

corpora_summary = {}
for corpus_id, rows in rows_by_corpus.items():
    speakers = {row.speaker for row in rows}
    hours = sum(row.duration_s for row in rows) / 3600
    corpora_summary[corpus_id] = {
        "sentences": len(rows),
        "speakers": len(speakers),
        "hours": round(hours, 2),
    }

model_records = []
for model in config.models:
    model_records.append({"id": model.id, "repo_id": model.repo_id, "revision": model.revision})

results = {
    "branch": BRANCH,
    "faster_whisper": FASTER_WHISPER_VERSION,
    "ctranslate2": CTRANSLATE2_VERSION,
    "cuda_libraries": CUDA_LIBRARIES,
    "python": platform.python_version(),
    "models": model_records,
    "decoding": {
        "language": config.language,
        "beam_size": config.beam_size,
        "temperature": config.temperature,
        "vad_filter": config.vad_filter,
        "condition_on_previous_text": config.condition_on_previous_text,
    },
    "bootstrap": {
        "draws": config.bootstrap_draws,
        "seed": config.seed,
        "unit": "speaker",
        "confidence_level": config.confidence_level,
    },
    "corpora": corpora_summary,
    "gpu_float16": results_by_model,
    "small_minus_base": small_minus_base,
    "int8_minus_float16": {
        "sentences": len(check_rows),
        "paths": check_paths,
        "models": int8_minus_float16,
    },
    "worst_sentences": worst,
    "suspect_references": suspect_references,
    "gpu_seconds": {model_id: round(seconds, 1) for model_id, seconds in gpu_seconds.items()},
    "latency_audio_paths": used_paths,
    "runtime_s": round(time.time() - NOTEBOOK_START),
}
results_path = OUTPUT_DIR / "c17_transcription.json"
results_path.write_text(json.dumps(results, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Écrit :", results_path, f"({results['runtime_s']} s au total)")